# Phase 2.3 - Reproducible Preprocessing Pipeline

## 1. Introduction and requirements

Requirement **2.3(A)** asks us to reuse training preprocessing on test data without
changing the steps. **2.3(B)** requires sklearn Pipeline. We reproduce the corrected
Phase 2.1 Standard baseline and apply both saved Phase 2.2 feature lists.
No models or Phase 3 performance comparisons are fitted here.

The assignment, Phase 1, Phase 2.1, Phase 2.2, selection JSON and source CSVs were
reviewed. Relevant IAU seminar notebooks are unavailable locally; no direct seminar
comparison is claimed. Run from the repository root or `notebooks/`, using the
project `.venv` kernel with pandas, NumPy, sklearn and joblib.
This notebook reads CSV/JSON inputs directly and does not execute other notebooks.

Contributions: Dmytro Petrenko [actual %]; Oleh Serdiukov [actual %].
Complete the actual percentages before submission.


In [1]:
from pathlib import Path
import hashlib
import json
import sys
import subprocess
import tempfile
import joblib
import pandas as pd
import numpy as np
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (OneHotEncoder, StandardScaler, MinMaxScaler,
                                   RobustScaler, PowerTransformer, QuantileTransformer)
from sklearn.model_selection import GroupShuffleSplit

DATA = Path("data") if Path("data").is_dir() else Path("../data")
if not DATA.is_dir():
    raise FileNotFoundError("Run from the repository root or notebooks/ with data/ available.")
PROCESSED = DATA / "processed"
REFERENCE_CSV = PROCESSED / "phase2_train_standard.csv"
SELECTION_JSON = PROCESSED / "phase2_2_selected_features.json"
SEED = 42
SPLIT_CONFIG = {"n_splits": 1, "test_size": 0.20, "random_state": SEED}
selection_spec = json.loads(SELECTION_JSON.read_text(encoding="utf-8"))
assert set(selection_spec["feature_subsets"]) == {"with_depot", "without_depot"}
assert hashlib.sha256(REFERENCE_CSV.read_bytes()).hexdigest() == selection_spec["source_sha256"],     "Phase 2.1 export differs from the one used for selection; investigate before continuing."
print("scikit-learn:", sklearn.__version__, "pandas:", pd.__version__, "NumPy:", np.__version__)


scikit-learn: 1.9.1 pandas: 3.0.6 NumPy: 2.5.4


## 2. Loading and integration of data

Use the same separators, exact-duplicate rule and source row IDs as Phase 2.1.
Validate U-prefixed identifiers before extracting numeric keys. A many-to-one
LEFT JOIN preserves unmatched readings and original labels. Gateway metadata is
not joined, consistent with the retained Phase 1/2.1 baseline.

Date parsing uses explicit formats and leaves ambiguous dates unresolved. Keep
pre-commissioning and negative-service-hour flags for audit; neither is a predictor.
No outlier removal or clipping is applied. These fixed rules learn no statistics,
so integration and audit flags can be created before the existing split.


In [2]:
readings_raw = pd.read_csv(DATA / "readings.csv", sep="\t")
readings = readings_raw.drop_duplicates().copy()
units = pd.read_csv(DATA / "units.csv")

readings.index.name = "reading_row_id"

ids = readings["unitId"].astype("string").str.strip()
assert ids.str.fullmatch(r"U[0-9]+", na=False).all()

readings["unit_key"] = pd.to_numeric(ids.str[1:]).astype("int64")
units["unit_key"] = pd.to_numeric(units["unitId"]).astype("int64")

assert readings[["unitId", "unit_key"]].drop_duplicates()["unit_key"].is_unique
assert units["unit_key"].notna().all() and units["unit_key"].is_unique
assert readings["Failure"].isin([0, 1]).all()

data = (
    readings.reset_index()
    .merge(
        units.drop(columns="unitId"),
        on="unit_key",
        how="left",
        validate="many_to_one",
        indicator="device_match",
        sort=False,
    )
    .set_index("reading_row_id")
)

data["unit_metadata_missing"] = data["device_match"].eq("left_only")

assert data.index.is_unique
assert len(data) == len(readings)
assert set(data.index) == set(readings.index)

pd.testing.assert_frame_equal(
    data[readings.columns].sort_index(),
    readings.sort_index(),
)
print(f"Rows after joining: {len(data):,}")
print("Join checks passed: readings and labels preserved.")
expected_hours = data["unit_key"].map(units.set_index("unit_key")["serviceHours"])
pd.testing.assert_series_equal(data["serviceHours"], expected_hours, check_names=False)
print(f"Exact duplicate rows removed: {len(readings_raw) - len(readings):,}")
print(f"Unmatched readings retained: {data['unit_metadata_missing'].sum():,}")


Rows after joining: 13,092
Join checks passed: readings and labels preserved.
Exact duplicate rows removed: 196
Unmatched readings retained: 131


In [3]:
def standardize_dates(series):
    text = series.astype("string").str.strip()
    parsed = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")

    #parse year-first dates and dates with named months
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%d %b %Y"):
        candidate = pd.to_datetime(text, format=fmt, errors="coerce")
        parsed = parsed.fillna(candidate)

    #compare both interpretations of numeric dates
    month_first = pd.to_datetime(
        text, format="%m/%d/%Y, %H:%M:%S", errors="coerce"
    )
    day_first = pd.to_datetime(
        text, format="%d/%m/%Y, %H:%M:%S", errors="coerce"
    )

    ambiguous = (
        month_first.notna()
        & day_first.notna()
        & month_first.ne(day_first)
    )

    numeric = month_first.fillna(day_first).mask(ambiguous)
    parsed = parsed.fillna(numeric)

    #distinguish unresolved dates from original missing values
    status = pd.Series("unparsed", index=series.index, dtype="string")
    status.loc[text.isna() | text.eq("")] = "missing"
    status.loc[parsed.notna()] = "parsed"
    status.loc[ambiguous] = "ambiguous"

    return pd.DataFrame({
        "parsed": parsed,
        "status": status,
        "has_time": text.str.contains(
            r"\d{2}:\d{2}:\d{2}", regex=True, na=False
        ),
    })

cycle_dates = standardize_dates(data["ts_cycle"])
for suffix in cycle_dates.columns:
    data[f"ts_cycle_{suffix}"] = cycle_dates[suffix]

eligible_dates = data["ts_cycle_parsed"].notna() & data["commission_year"].notna()
data["before_commissioning"] = (
    data["ts_cycle_parsed"].dt.year.lt(data["commission_year"])
    .where(eligible_dates).astype("boolean")
)
data["negative_serviceHours"] = (
    data["serviceHours"].lt(0).where(data["serviceHours"].notna()).astype("boolean")
)
# Dates and these audit flags are retained for review, not passed to the model.


## 3. Reproducing the train/test split

Replay the **same** GroupShuffleSplit configuration and source ordering as
Phase 2.1, not a newly chosen split. Check the expected rows and devices, complete
row coverage and disjoint device sets. The later full training-matrix comparison
also checks observation order and labels against the Phase 2.1 export.
The split audits below do not determine transformations or feature choices.


In [4]:
splitter = GroupShuffleSplit(**SPLIT_CONFIG)
train_idx, test_idx = next(splitter.split(data, groups=data["unit_key"]))
train = data.iloc[train_idx].copy()
test = data.iloc[test_idx].copy()
assert (len(train), len(test)) == (10512, 2580), "Investigate a Phase 2.1 split mismatch."
assert (train.unit_key.nunique(), test.unit_key.nunique()) == (1693, 424)
assert set(train.unit_key).isdisjoint(test.unit_key)
assert train.index.intersection(test.index).empty
pd.testing.assert_frame_equal(pd.concat([train, test]).sort_index(), data.sort_index())
assert set(train.Failure) == set(test.Failure) == {0, 1}
split_summary = pd.DataFrame([
    {"partition": name, "rows": len(part), "devices": part.unit_key.nunique(),
     "Failure_0": part.Failure.eq(0).sum(), "Failure_1": part.Failure.eq(1).sum()}
    for name, part in [("train", train), ("test", test)]
]).set_index("partition")
display(split_summary)
print("The original 80%/20% device split is reproduced with no shared devices or rows.")


,rows,devices,Failure_0,Failure_1
partition,,,,
train,10512,1693,9234,1278
test,2580,424,2264,316


The original 80%/20% device split is reproduced with no shared devices or rows.


## 4. Building the preprocessing Pipeline

**Fixed preparation:** work on copies, replace oil temperature -999 with missing,
retain its invalid flag, strip category whitespace without merging category codes,
and create the Phase 2.1 missingness flags. Raw joined data remains available.
Call the same `prepare_inputs` function for both partitions.

**Learned Pipeline:** ColumnTransformer sends numbers through median imputation
and StandardScaler, categories through dense one-hot encoding, and flags through
unchanged. Empty numeric training columns use the same zero fallback as Phase 2.1.
Unknown categories produce zeros in their fitted one-hot group without adding
columns. Identifiers, dates, audit columns and Failure are not selected as inputs.

Fixed preparation is intentionally outside sklearn: it learns nothing, and the
fitted artifact then contains only standard sklearn objects that can reload in a
fresh process. Its input contract is the output of `prepare_inputs`, not raw CSV
rows. There is no imputation or scaling before the train/test split.

**Preparation contract:** call `prepare_inputs` once on original joined values,
not on already prepared or scaled data. A second call would clear the invalid-oil
flag after -999 has already become missing. Keep the original joined inputs.
New batches must also preserve training category formats: depot 76 is encoded as
`"76.0"` in this dataset. An all-matched integer-typed batch would produce `"76"`,
which the encoder treats as unseen. Preserve the Phase 2.1 representation before
preparing such inputs; this function is not a general normalizer for arbitrary
new input formats. The current train/test workflow already preserves it.


In [5]:
numeric_features = [
    "motorCurrent",
    "filterDp",
    "temperatureOil",
    "pressure_reservoir",
    "vibrationRms",
    "flowmeter",
    "tempBearing",
    "discharge_dewpoint",
    "commission_year",
    "serviceHours",
]

categorical_features = [
    "codeLine",
    "class_duty",
    "id_depot",
    "owner_operator",
    "installRegion",
]

missing_features = [
    "filterDp",
    "vibrationRms",
    "temperatureOil",
    "serviceHours",
]

flag_features = [
    f"{column}_missing" for column in missing_features
] + ["unit_metadata_missing", "temperatureOil_invalid"]

def prepare_inputs(frame):
    frame = frame.copy()

    frame["temperatureOil_invalid"] = frame["temperatureOil"].eq(-999).astype("int8")
    frame["temperatureOil"] = frame["temperatureOil"].replace(-999, np.nan)

    for column in categorical_features:
        values = frame[column].astype("string").str.strip()
        frame[column] = values.replace("", pd.NA).fillna("missing").astype(str)

    for column in missing_features:
        frame[f"{column}_missing"] = frame[column].isna().astype("int8")

    frame["unit_metadata_missing"] = (
        frame["unit_metadata_missing"].astype("int8")
    )

    return frame

In [6]:
def make_preprocessor(numeric_transformer=None):
    if numeric_transformer is None:
        numeric_transformer = StandardScaler()
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("transformer", clone(numeric_transformer)),
    ])
    columns = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("flags", "passthrough", flag_features),
    ], remainder="drop", verbose_feature_names_out=False)
    return Pipeline([("columns", columns)]).set_output(transform="pandas")

# Keep the Phase 2.1 alternatives reusable, without repeating their experiments.
continuous_features = [c for c in numeric_features if c != "commission_year"]
def distribution_transformer(transformer):
    return ColumnTransformer([
        ("continuous", transformer, continuous_features),
        ("year", StandardScaler(), ["commission_year"]),
    ], verbose_feature_names_out=False)

alternative_transformers = {
    "MinMax": MinMaxScaler(), "Robust": RobustScaler(),
    "Yeo-Johnson": distribution_transformer(Pipeline([
        ("power", PowerTransformer(method="yeo-johnson", standardize=False)),
        ("scale", StandardScaler()),
    ])),
    "Quantile": distribution_transformer(QuantileTransformer(
        n_quantiles=200, output_distribution="normal", random_state=SEED)),
}
# Example for later experiments: make_preprocessor(alternative_transformers["Robust"])
# These alternatives are unfitted; Standard remains the selected baseline.
preprocessor = make_preprocessor()
display(preprocessor)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columns', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{

## 5. Fitting on training data

Separate Failure before preparing inputs. Fit the Pipeline once on training
inputs, then transform them. The imputer and scaler checks below use training
values only. Compare all 256 columns, every row and target label with the existing
Standard CSV; any mismatch stops execution rather than being accepted silently.
CSV conversion can produce last-digit rounding, so numerical equality uses a
small tolerance. Pandas may preserve integer flag types where the CSV uses floats.


In [7]:
X_train_raw = train.drop(columns="Failure")
y_train = train["Failure"].copy()
groups_train = train["unit_key"].copy()
X_train = prepare_inputs(X_train_raw)
assert "Failure" not in X_train
assert not {"Failure", "unitId", "unit_key", "serialNumber", "ts_cycle",
            "ts_cycle_parsed", "before_commissioning", "negative_serviceHours"} & set(
                numeric_features + categorical_features + flag_features)
preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
assert X_train_processed.index.equals(y_train.index)
assert X_train_processed.shape == (10512, 256)
assert X_train_processed.columns.is_unique

numeric_fitted = preprocessor.named_steps["columns"].named_transformers_["numeric"]
imputer = numeric_fitted.named_steps["imputer"]
scaler = numeric_fitted.named_steps["transformer"]
encoder = preprocessor.named_steps["columns"].named_transformers_["categorical"]
np.testing.assert_allclose(imputer.statistics_, X_train[numeric_features].median().fillna(0).to_numpy())
imputed_train = imputer.transform(X_train[numeric_features])
np.testing.assert_allclose(scaler.mean_, imputed_train.mean().to_numpy())
assert int(scaler.n_samples_seen_) == len(train)

reference = pd.read_csv(REFERENCE_CSV)
assert reference.shape == (10512, 257)
assert reference.drop(columns="Failure").columns.equals(X_train_processed.columns)
np.testing.assert_array_equal(reference.Failure.to_numpy(), y_train.to_numpy())
np.testing.assert_allclose(X_train_processed.to_numpy(dtype=float),
    reference.drop(columns="Failure").to_numpy(dtype=float), rtol=1e-10, atol=1e-10)
max_reference_difference = float(np.abs(X_train_processed.to_numpy(dtype=float) -
    reference.drop(columns="Failure").to_numpy(dtype=float)).max())
print("Phase 2.1 reproduction passed: all training rows, labels and feature names/order match.")
print(f"Maximum absolute CSV difference: {max_reference_difference:.3g}")
display(pd.DataFrame({"training_missing": X_train[numeric_features].isna().sum(),
    "fitted_median": imputer.statistics_, "fitted_mean": scaler.mean_,
    "fitted_scale": scaler.scale_}).round(4))


Phase 2.1 reproduction passed: all training rows, labels and feature names/order match.
Maximum absolute CSV difference: 8.88e-16


,training_missing,fitted_median,fitted_mean,fitted_scale
motorCurrent,0,5.0351,5.2562,3.6585
filterDp,436,21.3841,24.0145,11.1128
temperatureOil,31,54.8080,54.8488,12.0201
pressure_reservoir,0,8.7148,8.7116,0.6012
vibrationRms,420,2.7188,3.0472,1.5925
flowmeter,0,28.0728,28.0116,4.0122
tempBearing,0,65.0403,65.0165,8.0435
discharge_dewpoint,0,7.9881,7.9944,3.5041
commission_year,101,2021.0000,2020.5212,2.5181
serviceHours,600,11563.4000,13857.0388,8615.5744


## 6. Applying the fitted Pipeline to test data

Apply the same fixed preparation and call **transform only** on the fitted
Pipeline. Do not choose methods from test distributions. The model-state hash
covers learned medians, scales and encoder categories; it must stay unchanged.
Test labels are kept separately until export.


In [8]:
X_test_raw = test.drop(columns="Failure")
y_test = test["Failure"].copy()
groups_test = test["unit_key"].copy()
X_test = prepare_inputs(X_test_raw)
state_before_test = joblib.hash(preprocessor)
X_test_processed = preprocessor.transform(X_test)
assert joblib.hash(preprocessor) == state_before_test
assert X_test_processed.shape == (2580, 256)
assert X_test_processed.columns.equals(X_train_processed.columns)
assert X_test_processed.index.equals(y_test.index)
print("Test transformed without refitting; all 256 names and their order match train.")


Test transformed without refitting; all 256 names and their order match train.


## 7. Integrating the two feature subsets

Read the saved JSON lists in their exact order. A fitted, passthrough
ColumnTransformer selects names; it does not score features or learn from test.
Both candidate Pipelines share the **same fitted preprocessor**. Calling their
`transform` method prepares the full encoded matrix and selects the saved columns.
The input still needs the fixed `prepare_inputs` step described above.

Keep the broad 142-column and compact 22-column candidates. Do not add missing
columns or infer a new schema when a selected name is absent: fail and investigate.
These selections came from the full training set in Phase 2.2, so they are not
independent of future training-validation folds or automatically leakage-free.


In [9]:
subset_selectors = {}
subset_pipelines = {}
train_subsets = {}
test_subsets = {}
for name, definition in selection_spec["feature_subsets"].items():
    names = definition["selected_features"]
    assert len(names) == len(set(names)) == definition["column_count"]
    assert set(names) <= set(X_train_processed.columns), f"Missing selected columns for {name}."
    assert "Failure" not in names
    selector = ColumnTransformer([("selected", "passthrough", names)],
        remainder="drop", verbose_feature_names_out=False).set_output(transform="pandas")
    selector.fit(X_train_processed)
    subset_selectors[name] = selector
    subset_pipelines[name] = Pipeline([("preprocess", preprocessor), ("select", selector)])
    train_subsets[name] = selector.transform(X_train_processed)
    test_subsets[name] = subset_pipelines[name].transform(X_test)
    assert train_subsets[name].columns.to_list() == names
    assert test_subsets[name].columns.to_list() == names
    assert subset_pipelines[name].named_steps["preprocess"] is preprocessor
    pd.testing.assert_frame_equal(test_subsets[name], selector.transform(X_test_processed))
assert len(selection_spec["feature_subsets"]["with_depot"]["selected_features"]) == 142
assert len(selection_spec["feature_subsets"]["without_depot"]["selected_features"]) == 22
assert not any(c.startswith("id_depot_") for c in train_subsets["without_depot"])
assert joblib.hash(preprocessor) == state_before_test

schema_summary = pd.DataFrame([
    {"variant": name, "train_rows": len(a), "test_rows": len(b),
     "train_features": a.shape[1], "test_features": b.shape[1],
     "same_column_order": a.columns.equals(b.columns)}
    for name, a, b in [("Standard full", X_train_processed, X_test_processed)] +
        [(name, train_subsets[name], test_subsets[name]) for name in train_subsets]
]).set_index("variant")
display(schema_summary)


,train_rows,test_rows,train_features,test_features,same_column_order
variant,,,,,
Standard full,10512,2580,256,256,True
with_depot,10512,2580,142,142,True
without_depot,10512,2580,22,22,True


## 8. Validation and reproducibility checks

Check numeric, finite outputs, row/target alignment and disjoint devices for
both candidates. Repeat the same split only to verify determinism, not to select
a different partition. Refit a clone on the same training inputs to check repeatable
Standard output. This verification never fits on test data.

A two-row synthetic probe checks unseen and missing categories and distinguishes
invalid oil from originally missing oil. Unknown categories should produce zero
one-hot columns without changing the vocabulary or learned state.


In [10]:
frames = {"train_full": X_train_processed, "test_full": X_test_processed}
frames.update({f"train_{k}": v for k,v in train_subsets.items()})
frames.update({f"test_{k}": v for k,v in test_subsets.items()})
for name, frame in frames.items():
    assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in frame.dtypes)
    assert np.isfinite(frame.to_numpy(dtype=float)).all(), name
    expected_index = y_train.index if name.startswith("train_") else y_test.index
    assert frame.index.equals(expected_index), name
assert set(groups_train).isdisjoint(groups_test)

repeat_train_idx, repeat_test_idx = next(GroupShuffleSplit(**SPLIT_CONFIG).split(data, groups=data.unit_key))
np.testing.assert_array_equal(repeat_train_idx, train_idx)
np.testing.assert_array_equal(repeat_test_idx, test_idx)
repeat_preprocessor = clone(preprocessor).fit(X_train)
pd.testing.assert_frame_equal(repeat_preprocessor.transform(X_train), X_train_processed)
pd.testing.assert_frame_equal(preprocessor.transform(X_test), X_test_processed)

probe_raw = X_train_raw.iloc[:2].copy()
for column in categorical_features:
    probe_raw[column] = [f"__unseen_{column}__", None]
probe_raw["temperatureOil"] = [-999, np.nan]
probe = prepare_inputs(probe_raw)
assert probe.temperatureOil_invalid.to_list() == [1, 0]
assert probe.temperatureOil_missing.to_list() == [1, 1]
probe_processed = preprocessor.transform(probe)
assert probe_processed.columns.equals(X_train_processed.columns)
assert np.isfinite(probe_processed.to_numpy(dtype=float)).all()
for column in categorical_features:
    dummy_columns = [c for c in probe_processed if c.startswith(column + "_")]
    assert probe_processed.loc[probe_processed.index[0], dummy_columns].eq(0).all()
for name, pipeline in subset_pipelines.items():
    assert pipeline.transform(probe).columns.equals(train_subsets[name].columns)
assert joblib.hash(preprocessor) == state_before_test
print("PASS: numeric/finite outputs, group separation, target alignment, deterministic split and transforms.")
print("PASS: unseen/missing categories, distinct invalid/missing oil flags and unchanged fitted state.")


PASS: numeric/finite outputs, group separation, target alignment, deterministic split and transforms.
PASS: unseen/missing categories, distinct invalid/missing oil flags and unchanged fitted state.


## 9. Exporting processed datasets

Export separate train/test CSVs for both candidates, adding the unchanged Failure
label only after preprocessing. Export a row manifest with original reading IDs,
normalized unit keys, source unit IDs and per-partition CSV positions. This supplies
a **verified** device-group mapping that was missing from the Phase 2.1 CSV.

Save a fitted joblib bundle containing the full preprocessor, named selectors,
both candidate Pipelines and the saved feature definitions. It contains only
standard sklearn objects, so notebook-local function definitions are not needed
to unpickle it. For raw joined inputs, first apply `prepare_inputs` as shown here;
then use `bundle['pipelines'][name].transform(prepared_inputs)` for inference.
Load only locally trusted joblib artifacts and use compatible library versions.

Round-trip every CSV and reload the artifact in this kernel and a fresh Python
process. A JSON manifest records input hashes, package versions and output schemas.
These generated data/binary files are not staged for Git; avoid committing them.

The artifact is sufficient for correctly prepared inputs; it does not package raw
CSV integration or the external preparation function. Retain this notebook's
fixed preparation rules, input representations and compatible library versions.
The separate-process check uses three prepared test rows; it verifies artifact
loading and transformation, not ingestion of arbitrary new raw files.


In [11]:
PROCESSED.mkdir(parents=True, exist_ok=True)
export_rows = []
for name in train_subsets:
    for partition, features, target in [
        ("train", train_subsets[name], y_train), ("test", test_subsets[name], y_test)]:
        assert features.index.equals(target.index)
        exported = features.assign(Failure=target)
        output = PROCESSED / f"phase2_3_{partition}_{name}.csv"
        exported.to_csv(output, index=False)
        loaded = pd.read_csv(output)
        assert loaded.columns.to_list() == list(features.columns) + ["Failure"]
        assert loaded.shape == exported.shape
        assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in loaded.dtypes)
        assert np.isfinite(loaded.to_numpy(dtype=float)).all()
        np.testing.assert_allclose(loaded.to_numpy(dtype=float), exported.to_numpy(dtype=float), rtol=1e-10, atol=1e-10)
        np.testing.assert_array_equal(loaded.Failure, target.to_numpy())
        export_rows.append({"file": output.name, "rows": len(loaded),
                            "features": features.shape[1], "bytes": output.stat().st_size})

row_manifest = pd.concat([
    pd.DataFrame({"partition": name, "export_row": np.arange(len(part)),
        "reading_row_id": part.index.to_numpy(), "unitId": part.unitId.to_numpy(),
        "unit_key": part.unit_key.to_numpy()})
    for name,part in [("train", train), ("test", test)]
], ignore_index=True)
assert row_manifest.reading_row_id.is_unique
ROW_MANIFEST = PROCESSED / "phase2_3_split_rows.csv"
row_manifest.to_csv(ROW_MANIFEST, index=False)
restored_rows = pd.read_csv(ROW_MANIFEST)
for name,part in [("train",train),("test",test)]:
    restored_part = restored_rows.loc[restored_rows.partition.eq(name)].sort_values("export_row")
    np.testing.assert_array_equal(restored_part.reading_row_id, part.index)
    np.testing.assert_array_equal(restored_part.unit_key, part.unit_key.to_numpy())
    np.testing.assert_array_equal(restored_part.unitId, part.unitId.to_numpy())
assert set(restored_rows.loc[restored_rows.partition.eq("train"),"unit_key"]).isdisjoint(
    restored_rows.loc[restored_rows.partition.eq("test"),"unit_key"])

bundle = {"preprocessor": preprocessor, "subset_selectors": subset_selectors,
    "pipelines": subset_pipelines, "feature_subsets": selection_spec["feature_subsets"],
    "full_feature_names": X_train_processed.columns.to_list(),
    "input_contract": "Apply the fixed prepare_inputs function to joined non-target data before transform.",
    "numeric_features": numeric_features, "categorical_features": categorical_features,
    "flag_features": flag_features, "split_config": SPLIT_CONFIG,
    "sklearn_version": sklearn.__version__}
ARTIFACT = PROCESSED / "phase2_3_preprocessing.joblib"
joblib.dump(bundle, ARTIFACT)
restored_bundle = joblib.load(ARTIFACT)
for name in train_subsets:
    pd.testing.assert_frame_equal(restored_bundle["pipelines"][name].transform(X_test), test_subsets[name])

# A separate interpreter proves the artifact has no notebook-local pickle dependencies.
reload_script = """import sys, joblib, pandas as pd
bundle = joblib.load(sys.argv[1])
probe = pd.read_pickle(sys.argv[2])
outputs = {name: pipeline.transform(probe) for name,pipeline in bundle['pipelines'].items()}
joblib.dump(outputs, sys.argv[3])
"""
with tempfile.TemporaryDirectory(prefix="iau_phase23_reload_") as scratch:
    probe_file = Path(scratch) / "prepared_probe.pkl"
    result_file = Path(scratch) / "reload_result.joblib"
    X_test.iloc[:3].to_pickle(probe_file)
    subprocess.run([sys.executable, "-c", reload_script, str(ARTIFACT.resolve()),
                    str(probe_file), str(result_file)], check=True, capture_output=True, text=True)
    process_outputs = joblib.load(result_file)
    for name in train_subsets:
        pd.testing.assert_frame_equal(process_outputs[name], test_subsets[name].iloc[:3])

manifest = {
    "split_config": SPLIT_CONFIG, "split_rows_file": ROW_MANIFEST.name,
    "training_rows": len(train), "test_rows": len(test), "full_features": X_train_processed.shape[1],
    "phase21_max_abs_csv_difference": max_reference_difference,
    "phase22_selection_sha256": hashlib.sha256(SELECTION_JSON.read_bytes()).hexdigest(),
    "input_sha256": {name: hashlib.sha256((DATA/name).read_bytes()).hexdigest()
                     for name in ["readings.csv", "units.csv"]},
    "versions": {"sklearn": sklearn.__version__, "pandas": pd.__version__,
                 "numpy": np.__version__, "joblib": joblib.__version__},
    "artifact": ARTIFACT.name, "input_contract": bundle["input_contract"],
    "exports": export_rows,
}
MANIFEST = PROCESSED / "phase2_3_manifest.json"
MANIFEST.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
assert json.loads(MANIFEST.read_text())["split_config"] == SPLIT_CONFIG
assert joblib.hash(preprocessor) == state_before_test
print("CSV, row/group mapping and same-kernel/fresh-process artifact reloads passed.")
display(pd.DataFrame(export_rows))
print("Additional outputs:", ROW_MANIFEST.name, ARTIFACT.name, MANIFEST.name)


CSV, row/group mapping and same-kernel/fresh-process artifact reloads passed.


,file,rows,features,bytes
0,phase2_3_train_with_depot.csv,10512,142,7426980
1,phase2_3_test_with_depot.csv,2580,142,1824444
2,phase2_3_train_without_depot.csv,10512,22,2379530
3,phase2_3_test_without_depot.csv,2580,22,584354


Additional outputs: phase2_3_split_rows.csv phase2_3_preprocessing.joblib phase2_3_manifest.json


## 10. Conclusions and decisions for Phase 3

The same fitted Standard preprocessing serves train and test, with the saved
with-depot and without-depot feature schemas. Assertions and round trips verify
counts, names, order, labels and finite numeric outputs. Training values match
Phase 2.1 within CSV rounding tolerance. Deterministic cleaning remains outside
the learned Pipeline; this separation is the only structural change in processing.
These checks implement **2.3(A)** reuse and **2.3(B)** sklearn Pipeline.

The row manifest preserves verified groups for later validation. Use the original
prepared training inputs and groups, not the globally transformed CSVs, when
fitting preprocessing inside folds. The frozen Phase 2.2 lists also used all
training labels: refit data-driven feature selection inside folds before claiming
leakage-free validation. Merely cloning this frozen column selector is not enough.

Neither candidate has a demonstrated performance advantage. Missingness and
serviceHours may contain information recorded after Failure; metadata timing and
the prediction horizon remain unverified. Ambiguous dates, repeated readings and
shared depot conditions are retained limitations. Device holdout is not a verified
future-time or new-site forecasting evaluation. No test performance or Phase 3
model comparison is performed here.

**Merge note:** reuse one loading/integration block and the same split when
combining Phases 2.1-2.3. Reconcile repeated function and variable definitions:
Phase 2.1's `make_preprocessor()` defaults to median-only preprocessing, while
this notebook defaults to StandardScaler. Preserve the earlier comparisons and
make the final Standard choice explicit rather than relying on definition order.
